> **Strix — Drone Search & Rescue.** Part of the [Strix research repository](../README.md).
> This notebook covers **data preparation for the LADD detection dataset**. Full write-up in [`docs/`](../docs/).


# Hi all! 🙃
**In this work, you will learn more about how to work with Lacmus Drone Dataset (LADD). Data preparation will consist of just a few functions, so you can easily copy them and run them from your notebook so that you already have a dataset ready for work and immediately start working with neural network model.**

# Introduction

<div style="background-color:#d4f1f4; padding: 20px;">
<p style="font-size:20px; font-family:verdana; line-height: 1.7em">Volunteer search and rescue squad "LizaAlert" is a voluntary non-profit public association engaged in the search for missing people. The detachment was founded on October 15, 2010. According to the PSO, in 2019, 25,255 applications were received, of which 19,051 people were found alive and 2,043 dead by volunteers.</p>

<p style="font-size:20px; font-family:verdana; line-height: 1.7em"> The organization arose in September 2010, when a group of about five hundred spontaneously gathered volunteers searched for five-year-old Lisa Fomkina and her aunt who disappeared on September 13 in the vicinity of Orekhovo-Zuev near Moscow. The body of the girl was found 10 days later, the aunts the day before, both died of hypothermia. Volunteers who participated in the search came up with the idea to create a search and rescue team: in October 2010, the lizaalert.org website and forum were launched. The detachment was named "LizaAlert": in memory of the deceased girl and English. alert - "alarm signal".</p>

<p style="font-size:20px; font-family:verdana; line-height: 1.7em">In hopes of saving more lives, the rescue organization LizaAlert, along with volunteers and other organizations ("Sova"), began collecting data: drone shots of people. This should help to use modern detection architectures after training to search for missing people. Now you know history. You can join and try to improve the detection results or also start collecting data to merge them in the future</p>
</div>


![](https://upload.wikimedia.org/wikipedia/commons/thumb/8/8c/LizaAlert_logo2020.svg/250px-LizaAlert_logo2020.svg.png)

## Lacmus Drone Dataset (LADD)

**LADD is a dataset created by non-profit search and rescue volunteer organizations dedicated to finding missing persons (Owl, LisaAlert). The data is collected using drones and labeled with machine learning tools. The pictures were taken from a height of 40-50 meters, horizontally. The pictures depict people in various poses. The dataset consists of 1365 images. LADD annotations are available in VOC - Xmax, Ymax Xmin, Ymin and YOLO - XYWH formats.**  

#### Overview of dataset

* You can see a example of the image.

  We have just one kind of class:

  * 0 - `Human`


  ![example](https://i.ibb.co/GPqbG8Y/1020.jpg)

* The structure of the `LADD`

  ```
  ├── LADD
  │   ├── annotation
  │   │   └── VOC-format
  │   │        └── 0.xml
  │   │   └── YOLO-format
  │   │        └── 0.txt
  │   ├── description
  │   │   └── Description.md
  │   │   └── Description.pdf
  │   ├── image-sets
  │   │         # *.txt which split the dataset
  │   │      └──  test.txt
  │   │      └──  train.txt
  │   │      └──  trainval.txt
  │   │      └──  val.txt
  │   ├── images
  │   │   └── image.jpg
  ```


* The  `images`:
  * **Image Type** : *jpeg(JPEG)*
  * **Width** x **Height** : *4000 x 3000*


* The `annotation` : 
    1. The VOC format `.xml` for Object Detection, automatically generate by the label tools. Below is an example of `.xml` file.
    2. The YOLO format `.txt` for Object Detection, were obtained by extracting data from file.xml (VOC-format) , as well as converting it from XYXY to XYWH (YOLO-format). Below is an example of `.txt` file.
    

```
│   ├── xml [VOC] 
│   │   └── ...
│   ├── txt [YOLO]
│   │   └──   0    0.4195    0.9696    0.013    0.0213
│   │        Class   X          Y        W        H
```

The dataset is divided into 3 seasons: **summer**, **spring** and **winter**. 

# Preparing for data preprocessing ♻
1. **Importing libraries.**  
2. **Let's prepare the functions for preprocessing.**
3. **Let's do the preprocessing.**

## Libraires for data preparation
**And now we import all the necessary libraries. To prepare the data, we only need a couple of libraries. We will do all the rest of the work using the CLI, which is built into the frameworks we need.**

In [1]:
!python -m venv env
!pip install split-folders

In [2]:
import os
import shutil
import splitfolders
import pandas as pd
import numpy as np
from tqdm import tqdm
from colorama import Fore

## Pathes

In [3]:
IMAGE_PATH = "../input/lacmus-drone-dataset-ladd-v40/images/images" # The path to the folder with images.
TARGET_PATH = "../input/lacmus-drone-dataset-ladd-v40/annotation/annotation/YOLO-format" # The path to the folder with the annotation (labels).

# Helper functions for data preparation
* create_dataset() - Creates a dataframe with columns of paths paths to files with annotations and images
* prepare_dirs() - Creates folders in which files with images and their annotation in YOLO format will be extracted
* copy_dirs() - Copies files files with images and their angotation in YOLO format to separate folders
* finalizing_preparation() - Completes data processing. Checks the integrity of the dataset. Removes garbage after preprocessing

**Creates a dataframe with columns of paths paths to files with annotations and images**

In [4]:
def create_dataset(data_path: str, target_path: str) -> pd.DataFrame:
    assert isinstance(data_path, str) 
    assert isinstance(target_path, str)
    
    dict_paths = {
        "image": [],
        "annotation": []
    }
    
    for dir_name, _, filenames in os.walk(data_path):
        for filename in tqdm(filenames):
            name = filename.split('.')[0]
            dict_paths["image"].append(f"{data_path}/{name}.jpg")
            dict_paths["annotation"].append(f"{target_path}/{name}.txt")

    
    dataframe = pd.DataFrame(
        data=dict_paths,
        index=np.arange(0, len(dict_paths["image"]))
    )
    
    return dataframe

**Creates folders in which files with images and their annotation in YOLO format will be extracted**

In [5]:
def prepare_dirs(dataset_path: str,
                 annotation_path: str,
                 images_path: str) -> None:
    if not os.path.exists(dataset_path):
        os.mkdir(path=dataset_path)
        os.mkdir(path=annotation_path)
        os.mkdir(path=images_path)

**Copies files files with images and their angotation in YOLO format to separate folders**

In [6]:
def copy_dirs(dataframe: pd.DataFrame, 
             data_path: str,
             target_path: str) -> None:
    
    assert isinstance(dataframe, pd.DataFrame)
    assert isinstance(data_path, str) 
    assert isinstance(target_path, str)
    
    for i in tqdm(range(len(dataframe))):
        image_path, annotation_path = dataframe.iloc[i]
        shutil.copy(image_path, data_path)
        shutil.copy(annotation_path, target_path)

**Completes data processing. Checks the integrity of the dataset. Removes garbage after preprocessing**

In [7]:
def finalizing_preparation(dataset_path: str, ladd_path: str):
    assert os.path.exists(f"{dataset_path}")
    
    example_structure = [
        "dataset", 
        "train", "labels", "images",
        "test","labels", "images",
        "val", "labels", "images"
    ]
    
    dir_bone = (
        dirname.split("/")[-1]
        for dirname, _, filenames in os.walk('/kaggle/working/dataset')
        if dirname.split("/")[-1] in example_structure
    )
    
    try:
        print("\n~ Lacmus Dataset Structure ~\n")
        print(
        f"""
      ├── {next(dir_bone)}
      │   │
      │   ├── {next(dir_bone)}
      │   │   └── {next(dir_bone)}
      │   │   └── {next(dir_bone)}
      │   │        
      │   ├── {next(dir_bone)}
      │   │   └── {next(dir_bone)}
      │   │   └── {next(dir_bone)}
      │   │
      │   ├── {next(dir_bone)}
      │   │   └── {next(dir_bone)}
      │   │   └── {next(dir_bone)}
        """
        )
    except StopIteration as e:
        print(e)
    else:
        print(Fore.GREEN + "-> Success")
    finally:
        os.system(f"rm -rf {ladd_path}")

# Data preparation 🔩
**What do we need to do?** 
1. Create a dataframe with the columns 'image' and 'annotation', they will store the paths to the files with pictures and their annotation.
2. Create folders where we extract annotations in YOLO format and images
3. Then copy the files with pictures and their annotation to these folders
4. Use the library to split the dataset into training, test and validation samples. This format is not just convenient, but necessary for working with the ultralytics framework
5. Finish preprocessing. Check how the processes went

**We will use the YOLO format. The YOLO annotation is located in files with the txt extension.**

**Create a dataframe with the columns 'image' and 'annotation', they will store the paths to the files with pictures and their annotation.**

In [8]:
df = create_dataset(
    data_path=IMAGE_PATH,
    target_path=TARGET_PATH
)

100%|██████████| 1365/1365 [00:00<00:00, 863533.18it/s]


In [9]:
dataset_path = "../working/dataset"
ladd_path = "../working/ladd"
annotation_path = "../working/ladd/labels"
image_path = "../working/ladd/images" 

**Create folders where we extract annotations in YOLO format and images**

In [10]:
prepare_dirs(
    dataset_path=ladd_path,
    annotation_path=annotation_path,
    images_path=image_path
)

**Then copy the files with pictures and their annotation to these folders**

In [11]:
copy_dirs(
    dataframe=df, 
    data_path=image_path,
    target_path=annotation_path
)

100%|██████████| 1365/1365 [01:16<00:00, 17.74it/s]


**Use the library to split the dataset into training, test and validation samples. This format is not just convenient, but necessary for working with the ultralytics framework**

In [12]:
splitfolders.ratio(
    input=ladd_path,
    output=dataset_path,
    seed=42,
    ratio=(0.80, 0.10, 0.10),
    group_prefix=None,
    move=True
) 

Copying files: 2730 files [00:00, 20083.96 files/s]


**Finish preprocessing. Check how the processes went ✅**

In [13]:
finalizing_preparation(
    dataset_path,
    ladd_path
)


~ Lacmus Dataset Structure ~


      ├── dataset
      │   │
      │   ├── test
      │   │   └── images
      │   │   └── labels
      │   │        
      │   ├── train
      │   │   └── images
      │   │   └── labels
      │   │
      │   ├── val
      │   │   └── images
      │   │   └── labels
        
-> Success


# In custody ⚡️
**Thank you for choosing to check out the LADD dataset. I hope this work will give a better understanding of how to work with it. In general, you don't need to prepare the data yourself. Just copy the given functions and then run them. The data set is quite interesting and it is already enough for good results. Good luck 🙃**